In [ ]:
pip install lightgbm


   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------------------------------------ --- 1.3/1.5 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 6.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1 -> 26.0.1
[notice] To update, run: c:\Users\matja\anaconda3\python.exe -m pip install --upgrade pip


In [6]:
pip install lightgbm


   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------------------------------------ --- 1.3/1.5 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 6.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1 -> 26.0.1
[notice] To update, run: c:\Users\matja\anaconda3\python.exe -m pip install --upgrade pip


In [ ]:
"""
MARCH MACHINE LEARNING MANIA 2026
Dwa osobne modele: Men (od 1985) i Women (od 1998).
"""
 
import re
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
 
warnings.filterwarnings("ignore")
 
DATA_DIR     = Path("data")
SEASON       = 2026
TRAIN_FROM_M = 1985
TRAIN_FROM_W = 1998
COACH_WINDOW = 10   
 
def load(name):
    return pd.read_csv(DATA_DIR / f"{name}.csv")
 
def parse_seed(s):
    if pd.isna(s): return 16
    m = re.search(r"\d+", str(s))
    return int(m.group()) if m else 16
 
 

 
def get_season_stats(reg_detailed):
    """Statystyki per (Season, TeamID) z całego regularnego sezonu."""
    rows = []
    for side, opp in [("W", "L"), ("L", "W")]:
        df = reg_detailed.copy()
        r = pd.DataFrame()
        r["Season"] = df["Season"]
        r["TeamID"] = df[f"{side}TeamID"]
        r["Win"]    = 1 if side == "W" else 0
        r["Pts"]    = df[f"{side}Score"]
        r["OppPts"] = df[f"{opp}Score"]
        for c in ["FGM","FGA","FGM3","FGA3","FTM","FTA","OR","DR","TO","Stl","Blk"]:
            r[c]         = df[f"{side}{c}"].values if f"{side}{c}" in df.columns else 0.0
            r[f"Opp{c}"] = df[f"{opp}{c}"].values  if f"{opp}{c}"  in df.columns else 0.0
        rows.append(r)
 
    both = pd.concat(rows, ignore_index=True)
    agg = both.groupby(["Season","TeamID"]).agg(
        G=("Win","count"), Wins=("Win","sum"),
        Pts=("Pts","mean"), OppPts=("OppPts","mean"),
        FGM=("FGM","mean"), FGA=("FGA","mean"),
        FGM3=("FGM3","mean"), FGA3=("FGA3","mean"),
        FTM=("FTM","mean"), FTA=("FTA","mean"),
        OR=("OR","mean"), DR=("DR","mean"), TO=("TO","mean"),
        Stl=("Stl","mean"), Blk=("Blk","mean"),
        OppFGM=("OppFGM","mean"), OppFGA=("OppFGA","mean"),
        OppFGM3=("OppFGM3","mean"), OppFGA3=("OppFGA3","mean"),
    ).reset_index()
 
    agg["WinPct"]    = agg["Wins"] / agg["G"]
    agg["PtsDiff"]   = agg["Pts"] - agg["OppPts"]
    agg["eFGPct"]    = (agg["FGM"] + 0.5*agg["FGM3"]) / agg["FGA"].clip(lower=1)
    agg["OppeFGPct"] = (agg["OppFGM"] + 0.5*agg["OppFGM3"]) / agg["OppFGA"].clip(lower=1)
    agg["TOPct"]     = agg["TO"] / (agg["FGA"] + 0.44*agg["FTA"] + agg["TO"]).clip(lower=1)
    agg["ORPct"]     = agg["OR"] / (agg["OR"] + agg["DR"]).clip(lower=1)
    agg["FTRate"]    = agg["FTM"] / agg["FGA"].clip(lower=1)
    agg["Tempo"]     = agg["FGA"] - agg["OR"] + agg["TO"] + 0.44*agg["FTA"]
    return agg
 
 
def get_recent_form(reg_detailed, last_n=10):
    """Win% i PtsDiff z ostatnich last_n gier sezonu per (Season, TeamID)."""
    rows = []
    for side, opp in [("W", "L"), ("L", "W")]:
        df = reg_detailed[["Season","DayNum",f"{side}TeamID",
                            f"{side}Score",f"{opp}Score"]].copy()
        df.columns = ["Season","DayNum","TeamID","Pts","OppPts"]
        df["Win"] = 1 if side == "W" else 0
        rows.append(df)
    both = pd.concat(rows, ignore_index=True).sort_values(["Season","TeamID","DayNum"])
 
    form = (both.groupby(["Season","TeamID"])
                .apply(lambda g: g.tail(last_n)["Win"].mean())
                .reset_index(name="RecentWinPct"))
    pts  = (both.groupby(["Season","TeamID"])
                .apply(lambda g: (g.tail(last_n)["Pts"] -
                                   g.tail(last_n)["OppPts"]).mean())
                .reset_index(name="RecentPtsDiff"))
    return form.merge(pts, on=["Season","TeamID"])
 
 
def get_coach_stats(coaches_df, tour_compact, seed_map, current_season, window=10):

    if coaches_df is None or len(coaches_df) == 0:
        return pd.DataFrame(columns=["Season","TeamID",
                                      "CoachTourAppearances","CoachTourWinPct",
                                      "CoachAvgRoundReached","CoachTourWins"])

    tour = tour_compact.copy()

    def daynum_to_round(d):
        if d <= 136: return 1
        elif d <= 138: return 2
        elif d <= 143: return 3
        elif d <= 145: return 4
        elif d <= 152: return 5
        else: return 6
 
    tour["Round"] = tour["DayNum"].apply(daynum_to_round)
 
    wins_by_team = tour.groupby(["Season","WTeamID"]).agg(
        TourWins=("Round","count"),
        MaxRound=("Round","max")
    ).reset_index().rename(columns={"WTeamID":"TeamID"})
 
    all_teams = pd.concat([
        tour[["Season","WTeamID"]].rename(columns={"WTeamID":"TeamID"}),
        tour[["Season","LTeamID"]].rename(columns={"LTeamID":"TeamID"})
    ]).drop_duplicates()
    all_teams = all_teams.merge(wins_by_team, on=["Season","TeamID"], how="left")
    all_teams["TourWins"]  = all_teams["TourWins"].fillna(0)
    all_teams["MaxRound"]  = all_teams["MaxRound"].fillna(0)
    all_teams["TourGames"] = all_teams["TourWins"] + 1  
 

    coaches = coaches_df.copy()
    coaches = coaches[coaches["LastDayNum"] >= 134]  
 
    merged = all_teams.merge(coaches[["Season","TeamID","CoachName"]],
                              on=["Season","TeamID"], how="left")
    merged["CoachName"] = merged["CoachName"].fillna("unknown")
 
    results = []
    seasons_range = range(max(1985, current_season - window), current_season)
 
    coach_history = merged[merged["Season"].isin(seasons_range)].copy()
 
    coach_agg = coach_history.groupby("CoachName").agg(
        CoachTourAppearances = ("Season",    "count"),
        CoachTourWins        = ("TourWins",  "sum"),
        CoachTotalGames      = ("TourGames", "sum"),
        CoachAvgMaxRound     = ("MaxRound",  "mean"),
    ).reset_index()
    coach_agg["CoachTourWinPct"] = (coach_agg["CoachTourWins"] /
                                     coach_agg["CoachTotalGames"].clip(lower=1))
 
    current_coaches = coaches[coaches["Season"] == current_season][
        ["TeamID","CoachName"]].drop_duplicates("TeamID")
    current_coaches = current_coaches.merge(coach_agg, on="CoachName", how="left")
 

    current_coaches["CoachTourAppearances"] = current_coaches["CoachTourAppearances"].fillna(0)
    current_coaches["CoachTourWins"]        = current_coaches["CoachTourWins"].fillna(0)
    current_coaches["CoachTourWinPct"]      = current_coaches["CoachTourWinPct"].fillna(0.3)
    current_coaches["CoachAvgMaxRound"]     = current_coaches["CoachAvgMaxRound"].fillna(0.5)
 
    current_coaches["Season"] = current_season
    return current_coaches[["Season","TeamID","CoachTourAppearances",
                              "CoachTourWinPct","CoachAvgMaxRound","CoachTourWins"]]
 
 
def get_massey(massey_df):
    SYSTEMS = {"POM","SAG","MOR","RPI","WOL","DOK","KPK","BPI","NET","ESPN"}
    df = massey_df[massey_df["RankingDayNum"] <= 133].copy()
    if len(df) == 0:
        return pd.DataFrame(columns=["Season","TeamID","AvgRank"])
    avail = set(df["SystemName"].unique())
    use = list(SYSTEMS & avail) or list(avail)[:10]
    df = df[df["SystemName"].isin(use)]
    df = df.sort_values("RankingDayNum").groupby(["Season","SystemName","TeamID"]).last().reset_index()
    return df.groupby(["Season","TeamID"])["OrdinalRank"].mean().reset_index(name="AvgRank")
 
 
def get_seed_map(seeds_df):
    return {(int(r.Season), int(r.TeamID)): parse_seed(r.Seed)
            for r in seeds_df.itertuples()}
 
 
def get_seed_winrates(tour_compact, seed_map, before_season):
    rec = {s: [0, 0] for s in range(1, 17)}
    for r in tour_compact[tour_compact["Season"] < before_season].itertuples():
        ws = seed_map.get((r.Season, r.WTeamID), 16)
        ls = seed_map.get((r.Season, r.LTeamID), 16)
        rec[ws][0] += 1; rec[ws][1] += 1
        rec[ls][1] += 1
    return {s: v[0]/max(v[1],1) for s, v in rec.items()}
 
 
def get_tourney_history(tour_compact, seed_map, before_season):
    """Historyczny win% drużyny w turniejach — tylko z sezonów PRZED before_season."""
    rec = {}
    for r in tour_compact[tour_compact["Season"] < before_season].itertuples():
        for tid, won in [(r.WTeamID, 1), (r.LTeamID, 0)]:
            if tid not in rec: rec[tid] = [0, 0]
            rec[tid][0] += won; rec[tid][1] += 1
    return {tid: v[0]/v[1] for tid, v in rec.items() if v[1] > 0}
 
 

STAT_COLS = ["WinPct","PtsDiff","eFGPct","OppeFGPct",
             "TOPct","ORPct","FTRate","Tempo","Stl","Blk"]
 
COACH_COLS = ["CoachTourAppearances","CoachTourWinPct",
              "CoachAvgMaxRound","CoachTourWins"]
 
 
def make_row(t1, t2, season, stats, seed_map, massey, seed_wr,
             form, tourney_hist, coach_stats):
 
    def get_stat(tid):
        r = stats[(stats["Season"]==season) & (stats["TeamID"]==tid)]
        return r.iloc[0] if len(r) else None
 
    def get_rank(tid):
        r = massey[(massey["Season"]==season) & (massey["TeamID"]==tid)]
        return float(r.iloc[0]["AvgRank"]) if len(r) else 200.0
 
    def get_form(tid):
        r = form[(form["Season"]==season) & (form["TeamID"]==tid)]
        return r.iloc[0] if len(r) else None
 
    def get_coach(tid):
        r = coach_stats[(coach_stats["Season"]==season) & (coach_stats["TeamID"]==tid)]
        return r.iloc[0] if len(r) else None
 
    s1, s2 = get_stat(t1),  get_stat(t2)
    f1, f2 = get_form(t1),  get_form(t2)
    c1, c2 = get_coach(t1), get_coach(t2)
    seed1  = seed_map.get((season, t1), 16)
    seed2  = seed_map.get((season, t2), 16)
 
    feats = {
        # Seed
        "SeedDiff":        seed1 - seed2,
        "SeedWR":          seed_wr.get(seed1,.5) - seed_wr.get(seed2,.5),
        # Massey rankings
        "RankDiff":        get_rank(t1) - get_rank(t2),
        # Historia turniejowa drużyny
        "TourneyHistDiff": tourney_hist.get(t1,.5) - tourney_hist.get(t2,.5),
        # Forma ostatnich 10 gier
        "RecentWinPct":    (f1["RecentWinPct"]  if f1 is not None else .5) -
                           (f2["RecentWinPct"]  if f2 is not None else .5),
        "RecentPtsDiff":   (f1["RecentPtsDiff"] if f1 is not None else 0.) -
                           (f2["RecentPtsDiff"] if f2 is not None else 0.),
    }
 
    # Statystyki całego sezonu
    for col in STAT_COLS:
        v1 = float(s1[col]) if s1 is not None and col in s1.index else 0.0
        v2 = float(s2[col]) if s2 is not None and col in s2.index else 0.0
        feats[col] = v1 - v2
 
    # Doświadczenie trenera
    for col in COACH_COLS:
        v1 = float(c1[col]) if c1 is not None and col in c1.index else 0.0
        v2 = float(c2[col]) if c2 is not None and col in c2.index else 0.0
        feats[col] = v1 - v2
 
    return feats
 
 
def build_train(tour_compact, stats, seed_map, massey, seed_wr,
                form, tourney_hist, coach_stats_by_season, train_from):
    rows, labels = [], []
    df = tour_compact[(tour_compact["Season"] >= train_from) &
                      (tour_compact["Season"] <  SEASON)]
    for r in df.itertuples():
        season = int(r.Season)
        t1, t2 = min(r.WTeamID, r.LTeamID), max(r.WTeamID, r.LTeamID)
        cs = coach_stats_by_season.get(season, pd.DataFrame(
            columns=["Season","TeamID"]+COACH_COLS))
        rows.append(make_row(t1, t2, season, stats, seed_map, massey,
                              seed_wr, form, tourney_hist, cs))
        labels.append(1 if r.WTeamID == t1 else 0)
    return pd.DataFrame(rows).fillna(0), np.array(labels)
 
 
def build_test(sub_df, stats, seed_map, massey, seed_wr,
               form, tourney_hist, coach_stats):
    rows, ids = [], []
    for r in sub_df.itertuples():
        s, t1, t2 = r.ID.split("_")
        rows.append(make_row(int(t1), int(t2), int(s), stats, seed_map, massey,
                              seed_wr, form, tourney_hist, coach_stats))
        ids.append(r.ID)
    return ids, pd.DataFrame(rows).fillna(0)
 

 
def train_and_predict(X_train, y_train, X_test):
    from sklearn.linear_model  import LogisticRegression
    from sklearn.ensemble      import RandomForestClassifier, GradientBoostingClassifier
    from sklearn.calibration   import CalibratedClassifierCV
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline      import Pipeline
    from sklearn.model_selection import cross_val_score
    import xgboost  as xgb
    import lightgbm as lgb
 
    models = {
        "LR":  Pipeline([("sc", StandardScaler()),
                         ("m",  LogisticRegression(C=0.5, max_iter=1000, random_state=42))]),
        "XGB": CalibratedClassifierCV(
                   xgb.XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                                     subsample=0.8, colsample_bytree=0.8, reg_alpha=0.5,
                                     eval_metric="logloss", use_label_encoder=False,
                                     random_state=42, verbosity=0),
                   method="isotonic", cv=5),
        "LGB": CalibratedClassifierCV(
                   lgb.LGBMClassifier(n_estimators=300, num_leaves=16, learning_rate=0.05,
                                      subsample=0.8, colsample_bytree=0.8, reg_alpha=0.5,
                                      random_state=42, verbose=-1),
                   method="isotonic", cv=5),
        "RF":  CalibratedClassifierCV(
                   RandomForestClassifier(n_estimators=300, max_depth=5, min_samples_leaf=8,
                                          random_state=42, n_jobs=-1),
                   method="isotonic", cv=5),
        "GBM": CalibratedClassifierCV(
                   GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                                              subsample=0.8, random_state=42),
                   method="isotonic", cv=5),
    }
 
    print("  Cross-validation (5-fold Brier):")
    briers = {}
    for name, model in models.items():
        b = -cross_val_score(model, X_train, y_train, cv=5,
                             scoring="neg_brier_score", n_jobs=-1).mean()
        briers[name] = b
        bar = "█" * int((1 - b/0.25) * 15)
        print(f"    {name:<6} {b:.4f}  {bar}")
 
    inv   = {k: 1/v for k, v in briers.items()}
    total = sum(inv.values())
    w     = {k: v/total for k, v in inv.items()}
    print(f"  Wagi: { {k: f'{v:.3f}' for k,v in w.items()} }")
 
    prob = np.zeros(len(X_test))
    for name, model in models.items():
        model.fit(X_train, y_train)
        prob += w[name] * model.predict_proba(X_test)[:, 1]
 
    return np.clip(prob, 0.02, 0.98)
 
 
 
def run(label, reg_d, tour_c, seeds_df, massey_df, coaches_df, sub_df, train_from):
    print(f"\n{'='*55}")
    print(f"  {label}  (dane od {train_from})")
    print(f"{'='*55}")
 
    stats        = get_season_stats(reg_d)
    form         = get_recent_form(reg_d, last_n=10)
    seed_map     = get_seed_map(seeds_df)
    seed_wr      = get_seed_winrates(tour_c, seed_map, before_season=SEASON)
    massey       = get_massey(massey_df)
    tourney_hist = get_tourney_history(tour_c, seed_map, before_season=SEASON)
 

    all_seasons = sorted(tour_c["Season"].unique())
    coach_stats_by_season = {}
    for s in all_seasons:
        coach_stats_by_season[s] = get_coach_stats(
            coaches_df, tour_c, seed_map,
            current_season=s, window=COACH_WINDOW
        )

    coach_stats_2026 = get_coach_stats(
        coaches_df, tour_c, seed_map,
        current_season=SEASON, window=COACH_WINDOW
    )
 
    print(f"  Buduję zbiór treningowy ({train_from}–{SEASON-1})…")
    X_train, y_train = build_train(
        tour_c, stats, seed_map, massey, seed_wr,
        form, tourney_hist, coach_stats_by_season, train_from
    )
    print(f"  → {len(X_train):,} meczów | {X_train.shape[1]} cech | bilans {y_train.mean():.3f}")
 
    ids, X_test = build_test(
        sub_df, stats, seed_map, massey, seed_wr,
        form, tourney_hist, coach_stats_2026
    )
    for c in X_train.columns:
        if c not in X_test.columns:
            X_test[c] = 0.0
    X_test = X_test[X_train.columns]
 
    preds = train_and_predict(X_train, y_train, X_test)
    print(f"  → {len(preds):,} predykcji | śr: {preds.mean():.3f} | std: {preds.std():.3f}")
    return pd.DataFrame({"ID": ids, "Pred": preds})
 

 
if __name__ == "__main__":
    sub   = load("SampleSubmissionStage2")
    sub_M = sub[sub["ID"].apply(lambda x: int(x.split("_")[1]) < 3000)]
    sub_W = sub[sub["ID"].apply(lambda x: int(x.split("_")[1]) >= 3000)]
 
    pred_M = run(
        label      = "MĘŻCZYŹNI (TeamID 1xxx)",
        reg_d      = load("MRegularSeasonDetailedResults"),
        tour_c     = load("MNCAATourneyCompactResults"),
        seeds_df   = load("MNCAATourneySeeds"),
        massey_df  = pd.concat([load("MMasseyOrdinals_part_1"),
                                load("MMasseyOrdinals_part_2")], ignore_index=True),
        coaches_df = load("MTeamCoaches"),
        sub_df     = sub_M,
        train_from = TRAIN_FROM_M,
    )
 
    pred_W = run(
        label      = "KOBIETY (TeamID 3xxx)",
        reg_d      = load("WRegularSeasonDetailedResults"),
        tour_c     = load("WNCAATourneyCompactResults"),
        seeds_df   = load("WNCAATourneySeeds"),
        massey_df  = pd.DataFrame(columns=["Season","SystemName","TeamID",
                                            "RankingDayNum","OrdinalRank"]),
        coaches_df = None,   # brak danych o trenerach dla kobiet
        sub_df     = sub_W,
        train_from = TRAIN_FROM_W,
    )
 
    submission = pd.concat([pred_M, pred_W], ignore_index=True)
    submission.to_csv("submission.csv", index=False)
 
    print(f"\n{'='*55}")
    print(f"✅ submission.csv zapisany!")
    print(f"   Men:     {len(pred_M):,} predykcji")
    print(f"   Women:   {len(pred_W):,} predykcji")
    print(f"   Łącznie: {len(submission):,}")


  MĘŻCZYŹNI (TeamID 1xxx)  (dane od 1985)
  Buduję zbiór treningowy (1985–2025)…
  → 2,585 meczów | 20 cech | bilans 0.512
  Cross-validation (5-fold Brier):
    LR     0.1735  ████
    XGB    0.1769  ████
    LGB    0.1787  ████
    RF     0.1785  ████
    GBM    0.1762  ████
  Wagi: {'LR': '0.204', 'XGB': '0.200', 'LGB': '0.198', 'RF': '0.198', 'GBM': '0.201'}
  → 66,430 predykcji | śr: 0.439 | std: 0.262

  KOBIETY (TeamID 3xxx)  (dane od 1998)
  Buduję zbiór treningowy (1998–2025)…
  → 1,717 meczów | 20 cech | bilans 0.517
  Cross-validation (5-fold Brier):
    LR     0.1364  ██████
    XGB    0.1470  ██████
    LGB    0.1479  ██████
    RF     0.1413  ██████
    GBM    0.1437  ██████
  Wagi: {'LR': '0.210', 'XGB': '0.195', 'LGB': '0.194', 'RF': '0.203', 'GBM': '0.199'}
  → 65,703 predykcji | śr: 0.511 | std: 0.257

✅ submission.csv zapisany!
   Men:     66,430 predykcji
   Women:   65,703 predykcji
   Łącznie: 132,133


In [ ]:

# WALIDACJA NA SEZONIE 2025

 
import re
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
 
warnings.filterwarnings("ignore")
 
DATA_DIR     = Path("data")
TEST_YEAR    = 2025
TRAIN_FROM_M = 1985
TRAIN_FROM_W = 1998
COACH_WINDOW = 10
 
def load(name):
    return pd.read_csv(DATA_DIR / f"{name}.csv")
 
def parse_seed(s):
    if pd.isna(s): return 16
    m = re.search(r"\d+", str(s))
    return int(m.group()) if m else 16
 
def get_season_stats(reg_detailed):
    rows = []
    for side, opp in [("W", "L"), ("L", "W")]:
        df = reg_detailed.copy()
        r = pd.DataFrame()
        r["Season"] = df["Season"]
        r["TeamID"] = df[f"{side}TeamID"]
        r["Win"]    = 1 if side == "W" else 0
        r["Pts"]    = df[f"{side}Score"]
        r["OppPts"] = df[f"{opp}Score"]
        for c in ["FGM","FGA","FGM3","FGA3","FTM","FTA","OR","DR","TO","Stl","Blk"]:
            r[c]         = df[f"{side}{c}"].values if f"{side}{c}" in df.columns else 0.0
            r[f"Opp{c}"] = df[f"{opp}{c}"].values  if f"{opp}{c}"  in df.columns else 0.0
        rows.append(r)
    both = pd.concat(rows, ignore_index=True)
    agg = both.groupby(["Season","TeamID"]).agg(
        G=("Win","count"), Wins=("Win","sum"),
        Pts=("Pts","mean"), OppPts=("OppPts","mean"),
        FGM=("FGM","mean"), FGA=("FGA","mean"),
        FGM3=("FGM3","mean"), FGA3=("FGA3","mean"),
        FTM=("FTM","mean"), FTA=("FTA","mean"),
        OR=("OR","mean"), DR=("DR","mean"), TO=("TO","mean"),
        Stl=("Stl","mean"), Blk=("Blk","mean"),
        OppFGM=("OppFGM","mean"), OppFGA=("OppFGA","mean"),
        OppFGM3=("OppFGM3","mean"), OppFGA3=("OppFGA3","mean"),
    ).reset_index()
    agg["WinPct"]    = agg["Wins"] / agg["G"]
    agg["PtsDiff"]   = agg["Pts"] - agg["OppPts"]
    agg["eFGPct"]    = (agg["FGM"] + 0.5*agg["FGM3"]) / agg["FGA"].clip(lower=1)
    agg["OppeFGPct"] = (agg["OppFGM"] + 0.5*agg["OppFGM3"]) / agg["OppFGA"].clip(lower=1)
    agg["TOPct"]     = agg["TO"] / (agg["FGA"] + 0.44*agg["FTA"] + agg["TO"]).clip(lower=1)
    agg["ORPct"]     = agg["OR"] / (agg["OR"] + agg["DR"]).clip(lower=1)
    agg["FTRate"]    = agg["FTM"] / agg["FGA"].clip(lower=1)
    agg["Tempo"]     = agg["FGA"] - agg["OR"] + agg["TO"] + 0.44*agg["FTA"]
    return agg
 
def get_recent_form(reg_detailed, last_n=10):
    rows = []
    for side, opp in [("W", "L"), ("L", "W")]:
        df = reg_detailed[["Season","DayNum",f"{side}TeamID",
                            f"{side}Score",f"{opp}Score"]].copy()
        df.columns = ["Season","DayNum","TeamID","Pts","OppPts"]
        df["Win"] = 1 if side == "W" else 0
        rows.append(df)
    both = pd.concat(rows, ignore_index=True).sort_values(["Season","TeamID","DayNum"])
    form = (both.groupby(["Season","TeamID"])
                .apply(lambda g: g.tail(last_n)["Win"].mean())
                .reset_index(name="RecentWinPct"))
    pts  = (both.groupby(["Season","TeamID"])
                .apply(lambda g: (g.tail(last_n)["Pts"] -
                                   g.tail(last_n)["OppPts"]).mean())
                .reset_index(name="RecentPtsDiff"))
    return form.merge(pts, on=["Season","TeamID"])
 
def get_coach_stats(coaches_df, tour_compact, seed_map, current_season, window=10):
    if coaches_df is None or len(coaches_df) == 0:
        return pd.DataFrame(columns=["Season","TeamID","CoachTourAppearances",
                                      "CoachTourWinPct","CoachAvgMaxRound","CoachTourWins"])
    tour = tour_compact.copy()
    def daynum_to_round(d):
        if d <= 136: return 1
        elif d <= 138: return 2
        elif d <= 143: return 3
        elif d <= 145: return 4
        elif d <= 152: return 5
        else: return 6
    tour["Round"] = tour["DayNum"].apply(daynum_to_round)
    wins_by_team = tour.groupby(["Season","WTeamID"]).agg(
        TourWins=("Round","count"), MaxRound=("Round","max")
    ).reset_index().rename(columns={"WTeamID":"TeamID"})
    all_teams = pd.concat([
        tour[["Season","WTeamID"]].rename(columns={"WTeamID":"TeamID"}),
        tour[["Season","LTeamID"]].rename(columns={"LTeamID":"TeamID"})
    ]).drop_duplicates()
    all_teams = all_teams.merge(wins_by_team, on=["Season","TeamID"], how="left")
    all_teams["TourWins"]  = all_teams["TourWins"].fillna(0)
    all_teams["MaxRound"]  = all_teams["MaxRound"].fillna(0)
    all_teams["TourGames"] = all_teams["TourWins"] + 1
    coaches = coaches_df[coaches_df["LastDayNum"] >= 134].copy()
    merged = all_teams.merge(coaches[["Season","TeamID","CoachName"]],
                              on=["Season","TeamID"], how="left")
    merged["CoachName"] = merged["CoachName"].fillna("unknown")
    seasons_range = range(max(1985, current_season - window), current_season)
    coach_history = merged[merged["Season"].isin(seasons_range)]
    coach_agg = coach_history.groupby("CoachName").agg(
        CoachTourAppearances=("Season","count"),
        CoachTourWins=("TourWins","sum"),
        CoachTotalGames=("TourGames","sum"),
        CoachAvgMaxRound=("MaxRound","mean"),
    ).reset_index()
    coach_agg["CoachTourWinPct"] = (coach_agg["CoachTourWins"] /
                                     coach_agg["CoachTotalGames"].clip(lower=1))
    current_coaches = coaches[coaches["Season"] == current_season][
        ["TeamID","CoachName"]].drop_duplicates("TeamID")
    current_coaches = current_coaches.merge(coach_agg, on="CoachName", how="left")
    current_coaches["CoachTourAppearances"] = current_coaches["CoachTourAppearances"].fillna(0)
    current_coaches["CoachTourWins"]        = current_coaches["CoachTourWins"].fillna(0)
    current_coaches["CoachTourWinPct"]      = current_coaches["CoachTourWinPct"].fillna(0.3)
    current_coaches["CoachAvgMaxRound"]     = current_coaches["CoachAvgMaxRound"].fillna(0.5)
    current_coaches["Season"] = current_season
    return current_coaches[["Season","TeamID","CoachTourAppearances",
                              "CoachTourWinPct","CoachAvgMaxRound","CoachTourWins"]]
 
def get_massey(massey_df):
    SYSTEMS = {"POM","SAG","MOR","RPI","WOL","DOK","KPK","BPI","NET","ESPN"}
    df = massey_df[massey_df["RankingDayNum"] <= 133].copy()
    if len(df) == 0:
        return pd.DataFrame(columns=["Season","TeamID","AvgRank"])
    avail = set(df["SystemName"].unique())
    use = list(SYSTEMS & avail) or list(avail)[:10]
    df = df[df["SystemName"].isin(use)]
    df = df.sort_values("RankingDayNum").groupby(["Season","SystemName","TeamID"]).last().reset_index()
    return df.groupby(["Season","TeamID"])["OrdinalRank"].mean().reset_index(name="AvgRank")
 
def get_seed_map(seeds_df):
    return {(int(r.Season), int(r.TeamID)): parse_seed(r.Seed) for r in seeds_df.itertuples()}
 
def get_seed_winrates(tour_compact, seed_map, before_season):
    rec = {s: [0, 0] for s in range(1, 17)}
    for r in tour_compact[tour_compact["Season"] < before_season].itertuples():
        ws = seed_map.get((r.Season, r.WTeamID), 16)
        ls = seed_map.get((r.Season, r.LTeamID), 16)
        rec[ws][0] += 1; rec[ws][1] += 1
        rec[ls][1] += 1
    return {s: v[0]/max(v[1],1) for s, v in rec.items()}
 
def get_tourney_history(tour_compact, seed_map, before_season):
    """Historyczny win% drużyny w turniejach — tylko z sezonów PRZED before_season."""
    rec = {}
    for r in tour_compact[tour_compact["Season"] < before_season].itertuples():
        for tid, won in [(r.WTeamID, 1), (r.LTeamID, 0)]:
            if tid not in rec: rec[tid] = [0, 0]
            rec[tid][0] += won; rec[tid][1] += 1
    return {tid: v[0]/v[1] for tid, v in rec.items() if v[1] > 0}
 
STAT_COLS  = ["WinPct","PtsDiff","eFGPct","OppeFGPct","TOPct","ORPct","FTRate","Tempo","Stl","Blk"]
COACH_COLS = ["CoachTourAppearances","CoachTourWinPct","CoachAvgMaxRound","CoachTourWins"]
 
def make_row(t1, t2, season, stats, seed_map, massey, seed_wr,
             form, tourney_hist, coach_stats):
    def get_stat(tid):
        r = stats[(stats["Season"]==season) & (stats["TeamID"]==tid)]
        return r.iloc[0] if len(r) else None
    def get_rank(tid):
        r = massey[(massey["Season"]==season) & (massey["TeamID"]==tid)]
        return float(r.iloc[0]["AvgRank"]) if len(r) else 200.0
    def get_form(tid):
        r = form[(form["Season"]==season) & (form["TeamID"]==tid)]
        return r.iloc[0] if len(r) else None
    def get_coach(tid):
        r = coach_stats[(coach_stats["Season"]==season) & (coach_stats["TeamID"]==tid)]
        return r.iloc[0] if len(r) else None
 
    s1, s2 = get_stat(t1),  get_stat(t2)
    f1, f2 = get_form(t1),  get_form(t2)
    c1, c2 = get_coach(t1), get_coach(t2)
    seed1  = seed_map.get((season, t1), 16)
    seed2  = seed_map.get((season, t2), 16)
 
    feats = {
        "SeedDiff":        seed1 - seed2,
        "SeedWR":          seed_wr.get(seed1,.5) - seed_wr.get(seed2,.5),
        "RankDiff":        get_rank(t1) - get_rank(t2),
        "TourneyHistDiff": tourney_hist.get(t1,.5) - tourney_hist.get(t2,.5),
        "RecentWinPct":    (f1["RecentWinPct"]  if f1 is not None else .5) -
                           (f2["RecentWinPct"]  if f2 is not None else .5),
        "RecentPtsDiff":   (f1["RecentPtsDiff"] if f1 is not None else 0.) -
                           (f2["RecentPtsDiff"] if f2 is not None else 0.),
    }
    for col in STAT_COLS:
        v1 = float(s1[col]) if s1 is not None and col in s1.index else 0.0
        v2 = float(s2[col]) if s2 is not None and col in s2.index else 0.0
        feats[col] = v1 - v2
    for col in COACH_COLS:
        v1 = float(c1[col]) if c1 is not None and col in c1.index else 0.0
        v2 = float(c2[col]) if c2 is not None and col in c2.index else 0.0
        feats[col] = v1 - v2
    return feats
 
def get_models():
    from sklearn.linear_model  import LogisticRegression
    from sklearn.ensemble      import RandomForestClassifier, GradientBoostingClassifier
    from sklearn.calibration   import CalibratedClassifierCV
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline      import Pipeline
    import xgboost  as xgb
    import lightgbm as lgb
    return {
        "LR":  Pipeline([("sc", StandardScaler()),
                         ("m",  LogisticRegression(C=0.5, max_iter=1000, random_state=42))]),
        "XGB": CalibratedClassifierCV(
                   xgb.XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                                     subsample=0.8, colsample_bytree=0.8, reg_alpha=0.5,
                                     eval_metric="logloss", use_label_encoder=False,
                                     random_state=42, verbosity=0),
                   method="isotonic", cv=5),
        "LGB": CalibratedClassifierCV(
                   lgb.LGBMClassifier(n_estimators=300, num_leaves=16, learning_rate=0.05,
                                      subsample=0.8, colsample_bytree=0.8, reg_alpha=0.5,
                                      random_state=42, verbose=-1),
                   method="isotonic", cv=5),
        "RF":  CalibratedClassifierCV(
                   RandomForestClassifier(n_estimators=300, max_depth=5, min_samples_leaf=8,
                                          random_state=42, n_jobs=-1),
                   method="isotonic", cv=5),
        "GBM": CalibratedClassifierCV(
                   GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                                              subsample=0.8, random_state=42),
                   method="isotonic", cv=5),
    }
 
def validate(label, reg_d, tour_c, seeds_df, massey_df, coaches_df, train_from):
    print(f"\n{'='*55}")
    print(f"  {label}  |  trening od {train_from}  |  test: {TEST_YEAR}")
    print(f"{'='*55}")
 
    stats        = get_season_stats(reg_d)
    form         = get_recent_form(reg_d)
    seed_map     = get_seed_map(seeds_df)
    seed_wr      = get_seed_winrates(tour_c, seed_map, before_season=TEST_YEAR)
    massey       = get_massey(massey_df)
    tourney_hist = get_tourney_history(tour_c, seed_map, before_season=TEST_YEAR)
 
    all_seasons = sorted(tour_c["Season"].unique())
    coach_by_season = {}
    for s in all_seasons:
        coach_by_season[s] = get_coach_stats(
            coaches_df, tour_c, seed_map, current_season=s, window=COACH_WINDOW)
 
    train_df = tour_c[(tour_c["Season"] >= train_from) & (tour_c["Season"] < TEST_YEAR)]
    train_rows, train_labels = [], []
    for r in train_df.itertuples():
        season = int(r.Season)
        t1, t2 = min(r.WTeamID, r.LTeamID), max(r.WTeamID, r.LTeamID)
        cs = coach_by_season.get(season, pd.DataFrame(columns=["Season","TeamID"]+COACH_COLS))
        train_rows.append(make_row(t1, t2, season, stats, seed_map, massey,
                                    seed_wr, form, tourney_hist, cs))
        train_labels.append(1 if r.WTeamID == t1 else 0)
    X_train = pd.DataFrame(train_rows).fillna(0)
    y_train = np.array(train_labels)
    print(f"  Trening: {len(X_train):,} meczów ({train_from}–{TEST_YEAR-1})")

    cs_test  = coach_by_season.get(TEST_YEAR, pd.DataFrame(columns=["Season","TeamID"]+COACH_COLS))
    test_df  = tour_c[tour_c["Season"] == TEST_YEAR]
    test_rows, test_labels, test_ids = [], [], []
    for r in test_df.itertuples():
        t1, t2 = min(r.WTeamID, r.LTeamID), max(r.WTeamID, r.LTeamID)
        test_rows.append(make_row(t1, t2, TEST_YEAR, stats, seed_map, massey,
                                   seed_wr, form, tourney_hist, cs_test))
        test_labels.append(1 if r.WTeamID == t1 else 0)
        test_ids.append(f"{TEST_YEAR}_{t1}_{t2}")
    X_test = pd.DataFrame(test_rows).fillna(0)
    y_test = np.array(test_labels)
    for c in X_train.columns:
        if c not in X_test.columns: X_test[c] = 0.0
    X_test = X_test[X_train.columns]
    print(f"  Test:    {len(X_test)} meczów | {X_train.shape[1]} cech")
 
    from sklearn.model_selection import cross_val_score
    models = get_models()
 
    print(f"\n  {'MODEL':<8} {'CV Brier':>10} {'Test Brier':>11}")
    print(f"  {'─'*32}")
 
    briers_cv, trained = {}, {}
    for name, model in models.items():
        cv_b = -cross_val_score(model, X_train, y_train, cv=5,
                                scoring="neg_brier_score", n_jobs=-1).mean()
        model.fit(X_train, y_train)
        p      = np.clip(model.predict_proba(X_test)[:, 1], 0.02, 0.98)
        test_b = np.mean((p - y_test)**2)
        briers_cv[name] = cv_b
        trained[name]   = model
        print(f"  {name:<8} {cv_b:>10.4f} {test_b:>11.4f}")
 
    inv   = {k: 1/v for k, v in briers_cv.items()}
    total = sum(inv.values())
    w     = {k: v/total for k, v in inv.items()}
 
    ens = np.zeros(len(y_test))
    for name, model in trained.items():
        ens += w[name] * model.predict_proba(X_test)[:, 1]
    ens    = np.clip(ens, 0.02, 0.98)
    b_ens  = np.mean((ens - y_test)**2)
 
    base   = np.array([0.85 if make_row(
                int(i.split("_")[1]), int(i.split("_")[2]), TEST_YEAR,
                stats, seed_map, massey, seed_wr, form, tourney_hist,
                cs_test)["SeedDiff"] < 0 else 0.15 for i in test_ids])
    b_base = np.mean((base - y_test)**2)
 
    print(f"  {'─'*32}")
    print(f"  {'ENSEMBLE':<8} {'—':>10} {b_ens:>11.4f}")
    print(f"  {'BASELINE':<8} {'—':>10} {b_base:>11.4f}")
 
    results = pd.DataFrame({"ID": test_ids, "Pred": ens, "Actual": y_test})
    results["Error"] = np.abs(results["Pred"] - results["Actual"])
    print(f"\n  Top 5 upsetów:")
    print(f"  {'ID':<22} {'Pred':>6} {'Wynik':>10}")
    for _, row in results.sort_values("Error", ascending=False).head(5).iterrows():
        print(f"  {row['ID']:<22} {row['Pred']:>6.3f}"
              f" {'t1 wygrał' if row['Actual']==1 else 't2 wygrał':>10}")
 
    return b_ens, ens, y_test
 
if __name__ == "__main__":
    print("╔══════════════════════════════════════════════════════╗")
    print("║   WALIDACJA NA TURNIEJU 2025 — PORÓWNANIE MODELI     ║")
    print("╚══════════════════════════════════════════════════════╝")
 
    bM, pM, yM = validate(
        label="MĘŻCZYŹNI", train_from=TRAIN_FROM_M,
        reg_d      = load("MRegularSeasonDetailedResults"),
        tour_c     = load("MNCAATourneyCompactResults"),
        seeds_df   = load("MNCAATourneySeeds"),
        massey_df  = pd.concat([load("MMasseyOrdinals_part_1"),
                                load("MMasseyOrdinals_part_2")], ignore_index=True),
        coaches_df = load("MTeamCoaches"),
    )
 
    bW, pW, yW = validate(
        label="KOBIETY", train_from=TRAIN_FROM_W,
        reg_d      = load("WRegularSeasonDetailedResults"),
        tour_c     = load("WNCAATourneyCompactResults"),
        seeds_df   = load("WNCAATourneySeeds"),
        massey_df  = pd.DataFrame(columns=["Season","SystemName","TeamID",
                                            "RankingDayNum","OrdinalRank"]),
        coaches_df = None,
    )
 
    all_preds    = np.concatenate([pM, pW])
    all_labels   = np.concatenate([yM, yW])
    brier_kaggle = np.mean((all_preds - all_labels)**2)
 
    print(f"\n{'═'*55}")
    print(f"  PODSUMOWANIE KOŃCOWE — turniej {TEST_YEAR}")
    print(f"{'═'*55}")
    print(f"  {'':22} {'MĘŻCZYŹNI':>10} {'KOBIETY':>10}")
    print(f"  {'─'*44}")
    print(f"  {'Brier ensemble':<22} {bM:>10.4f} {bW:>10.4f}")
    print(f"  {'Liczba meczów':<22} {len(yM):>10} {len(yW):>10}")
    print(f"  {'─'*44}")
    print(f"  {'Brier jak Kaggle':<22} {brier_kaggle:>10.4f}")
    print(f"  {'Poprawa vs baseline':<22} {(1 - brier_kaggle/np.mean((np.full(len(all_labels),0.5) - all_labels)**2)):>9.1%}")
    print(f"\n  Dobry wynik na Kaggle: ~0.14–0.18")
    print(f"  Baseline (seed):       ~0.19–0.22")

╔══════════════════════════════════════════════════════╗
║   WALIDACJA NA TURNIEJU 2025 — PORÓWNANIE MODELI     ║
╚══════════════════════════════════════════════════════╝

  MĘŻCZYŹNI  |  trening od 1985  |  test: 2025
  Trening: 2,518 meczów (1985–2024)
  Test:    67 meczów | 20 cech

  MODEL      CV Brier  Test Brier
  ────────────────────────────────
  LR           0.1746      0.1639
  XGB          0.1787      0.1653
  LGB          0.1816      0.1730
  RF           0.1791      0.1561
  GBM          0.1778      0.1649
  ────────────────────────────────
  ENSEMBLE          —      0.1618
  BASELINE          —      0.1688

  Top 5 upsetów:
  ID                       Pred      Wynik
  2025_1155_1270          0.879  t2 wygrał
  2025_1166_1257          0.269  t1 wygrał
  2025_1140_1458          0.275  t1 wygrał
  2025_1116_1385          0.278  t1 wygrał
  2025_1279_1314          0.285  t1 wygrał

  KOBIETY  |  trening od 1998  |  test: 2025
  Trening: 1,650 meczów (1998–2024)
  Test:    67

In [ ]:

# WALIDACJA NA SEZONIE 2024 
 
import re
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
 
warnings.filterwarnings("ignore")
 
DATA_DIR     = Path("data")
TEST_YEAR    = 2024
TRAIN_FROM_M = 1985
TRAIN_FROM_W = 1998
COACH_WINDOW = 10
 
def load(name):
    return pd.read_csv(DATA_DIR / f"{name}.csv")
 
def parse_seed(s):
    if pd.isna(s): return 16
    m = re.search(r"\d+", str(s))
    return int(m.group()) if m else 16
 
def get_season_stats(reg_detailed):
    rows = []
    for side, opp in [("W", "L"), ("L", "W")]:
        df = reg_detailed.copy()
        r = pd.DataFrame()
        r["Season"] = df["Season"]
        r["TeamID"] = df[f"{side}TeamID"]
        r["Win"]    = 1 if side == "W" else 0
        r["Pts"]    = df[f"{side}Score"]
        r["OppPts"] = df[f"{opp}Score"]
        for c in ["FGM","FGA","FGM3","FGA3","FTM","FTA","OR","DR","TO","Stl","Blk"]:
            r[c]         = df[f"{side}{c}"].values if f"{side}{c}" in df.columns else 0.0
            r[f"Opp{c}"] = df[f"{opp}{c}"].values  if f"{opp}{c}"  in df.columns else 0.0
        rows.append(r)
    both = pd.concat(rows, ignore_index=True)
    agg = both.groupby(["Season","TeamID"]).agg(
        G=("Win","count"), Wins=("Win","sum"),
        Pts=("Pts","mean"), OppPts=("OppPts","mean"),
        FGM=("FGM","mean"), FGA=("FGA","mean"),
        FGM3=("FGM3","mean"), FGA3=("FGA3","mean"),
        FTM=("FTM","mean"), FTA=("FTA","mean"),
        OR=("OR","mean"), DR=("DR","mean"), TO=("TO","mean"),
        Stl=("Stl","mean"), Blk=("Blk","mean"),
        OppFGM=("OppFGM","mean"), OppFGA=("OppFGA","mean"),
        OppFGM3=("OppFGM3","mean"), OppFGA3=("OppFGA3","mean"),
    ).reset_index()
    agg["WinPct"]    = agg["Wins"] / agg["G"]
    agg["PtsDiff"]   = agg["Pts"] - agg["OppPts"]
    agg["eFGPct"]    = (agg["FGM"] + 0.5*agg["FGM3"]) / agg["FGA"].clip(lower=1)
    agg["OppeFGPct"] = (agg["OppFGM"] + 0.5*agg["OppFGM3"]) / agg["OppFGA"].clip(lower=1)
    agg["TOPct"]     = agg["TO"] / (agg["FGA"] + 0.44*agg["FTA"] + agg["TO"]).clip(lower=1)
    agg["ORPct"]     = agg["OR"] / (agg["OR"] + agg["DR"]).clip(lower=1)
    agg["FTRate"]    = agg["FTM"] / agg["FGA"].clip(lower=1)
    agg["Tempo"]     = agg["FGA"] - agg["OR"] + agg["TO"] + 0.44*agg["FTA"]
    return agg
 
def get_recent_form(reg_detailed, last_n=10):
    rows = []
    for side, opp in [("W", "L"), ("L", "W")]:
        df = reg_detailed[["Season","DayNum",f"{side}TeamID",
                            f"{side}Score",f"{opp}Score"]].copy()
        df.columns = ["Season","DayNum","TeamID","Pts","OppPts"]
        df["Win"] = 1 if side == "W" else 0
        rows.append(df)
    both = pd.concat(rows, ignore_index=True).sort_values(["Season","TeamID","DayNum"])
    form = (both.groupby(["Season","TeamID"])
                .apply(lambda g: g.tail(last_n)["Win"].mean())
                .reset_index(name="RecentWinPct"))
    pts  = (both.groupby(["Season","TeamID"])
                .apply(lambda g: (g.tail(last_n)["Pts"] -
                                   g.tail(last_n)["OppPts"]).mean())
                .reset_index(name="RecentPtsDiff"))
    return form.merge(pts, on=["Season","TeamID"])
 
def get_coach_stats(coaches_df, tour_compact, seed_map, current_season, window=10):
    if coaches_df is None or len(coaches_df) == 0:
        return pd.DataFrame(columns=["Season","TeamID","CoachTourAppearances",
                                      "CoachTourWinPct","CoachAvgMaxRound","CoachTourWins"])
    tour = tour_compact.copy()
    def daynum_to_round(d):
        if d <= 136: return 1
        elif d <= 138: return 2
        elif d <= 143: return 3
        elif d <= 145: return 4
        elif d <= 152: return 5
        else: return 6
    tour["Round"] = tour["DayNum"].apply(daynum_to_round)
    wins_by_team = tour.groupby(["Season","WTeamID"]).agg(
        TourWins=("Round","count"), MaxRound=("Round","max")
    ).reset_index().rename(columns={"WTeamID":"TeamID"})
    all_teams = pd.concat([
        tour[["Season","WTeamID"]].rename(columns={"WTeamID":"TeamID"}),
        tour[["Season","LTeamID"]].rename(columns={"LTeamID":"TeamID"})
    ]).drop_duplicates()
    all_teams = all_teams.merge(wins_by_team, on=["Season","TeamID"], how="left")
    all_teams["TourWins"]  = all_teams["TourWins"].fillna(0)
    all_teams["MaxRound"]  = all_teams["MaxRound"].fillna(0)
    all_teams["TourGames"] = all_teams["TourWins"] + 1
    coaches = coaches_df[coaches_df["LastDayNum"] >= 134].copy()
    merged = all_teams.merge(coaches[["Season","TeamID","CoachName"]],
                              on=["Season","TeamID"], how="left")
    merged["CoachName"] = merged["CoachName"].fillna("unknown")
    seasons_range = range(max(1985, current_season - window), current_season)
    coach_history = merged[merged["Season"].isin(seasons_range)]
    coach_agg = coach_history.groupby("CoachName").agg(
        CoachTourAppearances=("Season","count"),
        CoachTourWins=("TourWins","sum"),
        CoachTotalGames=("TourGames","sum"),
        CoachAvgMaxRound=("MaxRound","mean"),
    ).reset_index()
    coach_agg["CoachTourWinPct"] = (coach_agg["CoachTourWins"] /
                                     coach_agg["CoachTotalGames"].clip(lower=1))
    current_coaches = coaches[coaches["Season"] == current_season][
        ["TeamID","CoachName"]].drop_duplicates("TeamID")
    current_coaches = current_coaches.merge(coach_agg, on="CoachName", how="left")
    current_coaches["CoachTourAppearances"] = current_coaches["CoachTourAppearances"].fillna(0)
    current_coaches["CoachTourWins"]        = current_coaches["CoachTourWins"].fillna(0)
    current_coaches["CoachTourWinPct"]      = current_coaches["CoachTourWinPct"].fillna(0.3)
    current_coaches["CoachAvgMaxRound"]     = current_coaches["CoachAvgMaxRound"].fillna(0.5)
    current_coaches["Season"] = current_season
    return current_coaches[["Season","TeamID","CoachTourAppearances",
                              "CoachTourWinPct","CoachAvgMaxRound","CoachTourWins"]]
 
def get_massey(massey_df):
    SYSTEMS = {"POM","SAG","MOR","RPI","WOL","DOK","KPK","BPI","NET","ESPN"}
    df = massey_df[massey_df["RankingDayNum"] <= 133].copy()
    if len(df) == 0:
        return pd.DataFrame(columns=["Season","TeamID","AvgRank"])
    avail = set(df["SystemName"].unique())
    use = list(SYSTEMS & avail) or list(avail)[:10]
    df = df[df["SystemName"].isin(use)]
    df = df.sort_values("RankingDayNum").groupby(["Season","SystemName","TeamID"]).last().reset_index()
    return df.groupby(["Season","TeamID"])["OrdinalRank"].mean().reset_index(name="AvgRank")
 
def get_seed_map(seeds_df):
    return {(int(r.Season), int(r.TeamID)): parse_seed(r.Seed) for r in seeds_df.itertuples()}
 
def get_seed_winrates(tour_compact, seed_map, before_season):
    rec = {s: [0, 0] for s in range(1, 17)}
    for r in tour_compact[tour_compact["Season"] < before_season].itertuples():
        ws = seed_map.get((r.Season, r.WTeamID), 16)
        ls = seed_map.get((r.Season, r.LTeamID), 16)
        rec[ws][0] += 1; rec[ws][1] += 1
        rec[ls][1] += 1
    return {s: v[0]/max(v[1],1) for s, v in rec.items()}
 
def get_tourney_history(tour_compact, seed_map, before_season):
    """Historyczny win% drużyny w turniejach — tylko z sezonów PRZED before_season."""
    rec = {}
    for r in tour_compact[tour_compact["Season"] < before_season].itertuples():
        for tid, won in [(r.WTeamID, 1), (r.LTeamID, 0)]:
            if tid not in rec: rec[tid] = [0, 0]
            rec[tid][0] += won; rec[tid][1] += 1
    return {tid: v[0]/v[1] for tid, v in rec.items() if v[1] > 0}
 
STAT_COLS  = ["WinPct","PtsDiff","eFGPct","OppeFGPct","TOPct","ORPct","FTRate","Tempo","Stl","Blk"]
COACH_COLS = ["CoachTourAppearances","CoachTourWinPct","CoachAvgMaxRound","CoachTourWins"]
 
def make_row(t1, t2, season, stats, seed_map, massey, seed_wr,
             form, tourney_hist, coach_stats):
    def get_stat(tid):
        r = stats[(stats["Season"]==season) & (stats["TeamID"]==tid)]
        return r.iloc[0] if len(r) else None
    def get_rank(tid):
        r = massey[(massey["Season"]==season) & (massey["TeamID"]==tid)]
        return float(r.iloc[0]["AvgRank"]) if len(r) else 200.0
    def get_form(tid):
        r = form[(form["Season"]==season) & (form["TeamID"]==tid)]
        return r.iloc[0] if len(r) else None
    def get_coach(tid):
        r = coach_stats[(coach_stats["Season"]==season) & (coach_stats["TeamID"]==tid)]
        return r.iloc[0] if len(r) else None
 
    s1, s2 = get_stat(t1),  get_stat(t2)
    f1, f2 = get_form(t1),  get_form(t2)
    c1, c2 = get_coach(t1), get_coach(t2)
    seed1  = seed_map.get((season, t1), 16)
    seed2  = seed_map.get((season, t2), 16)
 
    feats = {
        "SeedDiff":        seed1 - seed2,
        "SeedWR":          seed_wr.get(seed1,.5) - seed_wr.get(seed2,.5),
        "RankDiff":        get_rank(t1) - get_rank(t2),
        "TourneyHistDiff": tourney_hist.get(t1,.5) - tourney_hist.get(t2,.5),
        "RecentWinPct":    (f1["RecentWinPct"]  if f1 is not None else .5) -
                           (f2["RecentWinPct"]  if f2 is not None else .5),
        "RecentPtsDiff":   (f1["RecentPtsDiff"] if f1 is not None else 0.) -
                           (f2["RecentPtsDiff"] if f2 is not None else 0.),
    }
    for col in STAT_COLS:
        v1 = float(s1[col]) if s1 is not None and col in s1.index else 0.0
        v2 = float(s2[col]) if s2 is not None and col in s2.index else 0.0
        feats[col] = v1 - v2
    for col in COACH_COLS:
        v1 = float(c1[col]) if c1 is not None and col in c1.index else 0.0
        v2 = float(c2[col]) if c2 is not None and col in c2.index else 0.0
        feats[col] = v1 - v2
    return feats
 
def get_models():
    from sklearn.linear_model  import LogisticRegression
    from sklearn.ensemble      import RandomForestClassifier, GradientBoostingClassifier
    from sklearn.calibration   import CalibratedClassifierCV
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline      import Pipeline
    import xgboost  as xgb
    import lightgbm as lgb
    return {
        "LR":  Pipeline([("sc", StandardScaler()),
                         ("m",  LogisticRegression(C=0.5, max_iter=1000, random_state=42))]),
        "XGB": CalibratedClassifierCV(
                   xgb.XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                                     subsample=0.8, colsample_bytree=0.8, reg_alpha=0.5,
                                     eval_metric="logloss", use_label_encoder=False,
                                     random_state=42, verbosity=0),
                   method="isotonic", cv=5),
        "LGB": CalibratedClassifierCV(
                   lgb.LGBMClassifier(n_estimators=300, num_leaves=16, learning_rate=0.05,
                                      subsample=0.8, colsample_bytree=0.8, reg_alpha=0.5,
                                      random_state=42, verbose=-1),
                   method="isotonic", cv=5),
        "RF":  CalibratedClassifierCV(
                   RandomForestClassifier(n_estimators=300, max_depth=5, min_samples_leaf=8,
                                          random_state=42, n_jobs=-1),
                   method="isotonic", cv=5),
        "GBM": CalibratedClassifierCV(
                   GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                                              subsample=0.8, random_state=42),
                   method="isotonic", cv=5),
    }
 
def validate(label, reg_d, tour_c, seeds_df, massey_df, coaches_df, train_from):
    print(f"\n{'='*55}")
    print(f"  {label}  |  trening od {train_from}  |  test: {TEST_YEAR}")
    print(f"{'='*55}")
 
    stats        = get_season_stats(reg_d)
    form         = get_recent_form(reg_d)
    seed_map     = get_seed_map(seeds_df)
    seed_wr      = get_seed_winrates(tour_c, seed_map, before_season=TEST_YEAR)
    massey       = get_massey(massey_df)
    tourney_hist = get_tourney_history(tour_c, seed_map, before_season=TEST_YEAR)
 
    all_seasons = sorted(tour_c["Season"].unique())
    coach_by_season = {}
    for s in all_seasons:
        coach_by_season[s] = get_coach_stats(
            coaches_df, tour_c, seed_map, current_season=s, window=COACH_WINDOW)

    train_df = tour_c[(tour_c["Season"] >= train_from) & (tour_c["Season"] < TEST_YEAR)]
    train_rows, train_labels = [], []
    for r in train_df.itertuples():
        season = int(r.Season)
        t1, t2 = min(r.WTeamID, r.LTeamID), max(r.WTeamID, r.LTeamID)
        cs = coach_by_season.get(season, pd.DataFrame(columns=["Season","TeamID"]+COACH_COLS))
        train_rows.append(make_row(t1, t2, season, stats, seed_map, massey,
                                    seed_wr, form, tourney_hist, cs))
        train_labels.append(1 if r.WTeamID == t1 else 0)
    X_train = pd.DataFrame(train_rows).fillna(0)
    y_train = np.array(train_labels)
    print(f"  Trening: {len(X_train):,} meczów ({train_from}–{TEST_YEAR-1})")
 

    cs_test  = coach_by_season.get(TEST_YEAR, pd.DataFrame(columns=["Season","TeamID"]+COACH_COLS))
    test_df  = tour_c[tour_c["Season"] == TEST_YEAR]
    test_rows, test_labels, test_ids = [], [], []
    for r in test_df.itertuples():
        t1, t2 = min(r.WTeamID, r.LTeamID), max(r.WTeamID, r.LTeamID)
        test_rows.append(make_row(t1, t2, TEST_YEAR, stats, seed_map, massey,
                                   seed_wr, form, tourney_hist, cs_test))
        test_labels.append(1 if r.WTeamID == t1 else 0)
        test_ids.append(f"{TEST_YEAR}_{t1}_{t2}")
    X_test = pd.DataFrame(test_rows).fillna(0)
    y_test = np.array(test_labels)
    for c in X_train.columns:
        if c not in X_test.columns: X_test[c] = 0.0
    X_test = X_test[X_train.columns]
    print(f"  Test:    {len(X_test)} meczów | {X_train.shape[1]} cech")
 
    from sklearn.model_selection import cross_val_score
    models = get_models()
 
    print(f"\n  {'MODEL':<8} {'CV Brier':>10} {'Test Brier':>11}")
    print(f"  {'─'*32}")
 
    briers_cv, trained = {}, {}
    for name, model in models.items():
        cv_b = -cross_val_score(model, X_train, y_train, cv=5,
                                scoring="neg_brier_score", n_jobs=-1).mean()
        model.fit(X_train, y_train)
        p      = np.clip(model.predict_proba(X_test)[:, 1], 0.02, 0.98)
        test_b = np.mean((p - y_test)**2)
        briers_cv[name] = cv_b
        trained[name]   = model
        print(f"  {name:<8} {cv_b:>10.4f} {test_b:>11.4f}")
 
    inv   = {k: 1/v for k, v in briers_cv.items()}
    total = sum(inv.values())
    w     = {k: v/total for k, v in inv.items()}
 
    ens = np.zeros(len(y_test))
    for name, model in trained.items():
        ens += w[name] * model.predict_proba(X_test)[:, 1]
    ens    = np.clip(ens, 0.02, 0.98)
    b_ens  = np.mean((ens - y_test)**2)
 
    base   = np.array([0.85 if make_row(
                int(i.split("_")[1]), int(i.split("_")[2]), TEST_YEAR,
                stats, seed_map, massey, seed_wr, form, tourney_hist,
                cs_test)["SeedDiff"] < 0 else 0.15 for i in test_ids])
    b_base = np.mean((base - y_test)**2)
 
    print(f"  {'─'*32}")
    print(f"  {'ENSEMBLE':<8} {'—':>10} {b_ens:>11.4f}")
    print(f"  {'BASELINE':<8} {'—':>10} {b_base:>11.4f}")
 
    results = pd.DataFrame({"ID": test_ids, "Pred": ens, "Actual": y_test})
    results["Error"] = np.abs(results["Pred"] - results["Actual"])
    print(f"\n  Top 5 upsetów:")
    print(f"  {'ID':<22} {'Pred':>6} {'Wynik':>10}")
    for _, row in results.sort_values("Error", ascending=False).head(5).iterrows():
        print(f"  {row['ID']:<22} {row['Pred']:>6.3f}"
              f" {'t1 wygrał' if row['Actual']==1 else 't2 wygrał':>10}")
 
    return b_ens, ens, y_test
 
if __name__ == "__main__":
    print("╔══════════════════════════════════════════════════════╗")
    print("║   WALIDACJA NA TURNIEJU 2024 — PORÓWNANIE MODELI     ║")
    print("╚══════════════════════════════════════════════════════╝")
 
    bM, pM, yM = validate(
        label="MĘŻCZYŹNI", train_from=TRAIN_FROM_M,
        reg_d      = load("MRegularSeasonDetailedResults"),
        tour_c     = load("MNCAATourneyCompactResults"),
        seeds_df   = load("MNCAATourneySeeds"),
        massey_df  = pd.concat([load("MMasseyOrdinals_part_1"),
                                load("MMasseyOrdinals_part_2")], ignore_index=True),
        coaches_df = load("MTeamCoaches"),
    )
 
    bW, pW, yW = validate(
        label="KOBIETY", train_from=TRAIN_FROM_W,
        reg_d      = load("WRegularSeasonDetailedResults"),
        tour_c     = load("WNCAATourneyCompactResults"),
        seeds_df   = load("WNCAATourneySeeds"),
        massey_df  = pd.DataFrame(columns=["Season","SystemName","TeamID",
                                            "RankingDayNum","OrdinalRank"]),
        coaches_df = None,
    )
 
    all_preds    = np.concatenate([pM, pW])
    all_labels   = np.concatenate([yM, yW])
    brier_kaggle = np.mean((all_preds - all_labels)**2)
 
    print(f"\n{'═'*55}")
    print(f"  PODSUMOWANIE KOŃCOWE — turniej {TEST_YEAR}")
    print(f"{'═'*55}")
    print(f"  {'':22} {'MĘŻCZYŹNI':>10} {'KOBIETY':>10}")
    print(f"  {'─'*44}")
    print(f"  {'Brier ensemble':<22} {bM:>10.4f} {bW:>10.4f}")
    print(f"  {'Liczba meczów':<22} {len(yM):>10} {len(yW):>10}")
    print(f"  {'─'*44}")
    print(f"  {'Brier jak Kaggle':<22} {brier_kaggle:>10.4f}")
    print(f"  {'Poprawa vs baseline':<22} {(1 - brier_kaggle/np.mean((np.full(len(all_labels),0.5) - all_labels)**2)):>9.1%}")
    print(f"\n  Dobry wynik na Kaggle: ~0.14–0.18")
    print(f"  Baseline (seed):       ~0.19–0.22")
 

╔══════════════════════════════════════════════════════╗
║   WALIDACJA NA TURNIEJU 2024 — PORÓWNANIE MODELI     ║
╚══════════════════════════════════════════════════════╝

  MĘŻCZYŹNI  |  trening od 1985  |  test: 2024
  Trening: 2,451 meczów (1985–2023)
  Test:    67 meczów | 20 cech

  MODEL      CV Brier  Test Brier
  ────────────────────────────────
  LR           0.1745      0.1952
  XGB          0.1774      0.1932
  LGB          0.1814      0.1899
  RF           0.1788      0.1944
  GBM          0.1780      0.1987
  ────────────────────────────────
  ENSEMBLE          —      0.1912
  BASELINE          —      0.2524

  Top 5 upsetów:
  ID                       Pred      Wynik
  2024_1246_1324          0.940  t2 wygrał
  2024_1120_1463          0.853  t2 wygrał
  2024_1160_1196          0.157  t1 wygrał
  2024_1112_1155          0.816  t2 wygrał
  2024_1213_1388          0.189  t1 wygrał

  KOBIETY  |  trening od 1998  |  test: 2024
  Trening: 1,583 meczów (1998–2023)
  Test:    67